# 02 — Conditional OFI Interaction

Primary hypothesis test: confirmed-event 1-min IC > 0.
Conditional OFI regression with HAC and clustered SEs, Wald test b_pos==b_neg.

In [1]:
import sys, os
from pathlib import Path
for _cand in ['.', '..']:
    if (Path(_cand)/'src').is_dir() and (Path(_cand)/'legacy').is_dir():
        os.chdir(_cand); break
sys.path.insert(0, 'src'); sys.path.insert(0, 'legacy/src')

import warnings; warnings.filterwarnings('ignore')
import numpy as np
import pandas as pd
import matplotlib; matplotlib.use('Agg')
import matplotlib.pyplot as plt

from src.stats_rigor import bootstrap_ic_ci, spearman_ic
from src.regression import fit_all_horizons
from src.grid import append_cells
from src.report_io import save_fig, save_table, setup_style
from src.config import PANELS_2016, COLORS

setup_style()
N_BOOT   = 1000
HORIZONS = [1, 5, 15]
print('Setup complete.')

Setup complete.


## 1. Load pooled 2016-2020 panel

In [2]:
frames = []
for ticker, path in PANELS_2016.items():
    if not Path(path).exists():
        print(f'  skip {ticker}')
        continue
    df = pd.read_csv(path)
    df['stock'] = ticker
    frames.append(df)

if not frames:
    raise RuntimeError('No 2016-2020 panels found — check PANELS_2016 in src/config.py')

panel = pd.concat(frames, ignore_index=True)
print(f'Pooled panel: {len(panel):,} events, columns: {list(panel.columns)}')

Pooled panel: 12,456 events, columns: ['timestamp', 'headline', 'stock', 'source', 'date', 'time', 'lm_score', 'llm_score', 'bar_time', 'ofi', 'ofi_z', 'mid', 'ret_1m', 'ret_5m', 'ret_15m']


## 2. Confirmed vs conflicted split

In [3]:
# Confirmed: sign(llm_score) == sign(ofi_z), both non-zero
same_sign = (
    (panel['llm_score'] != 0) &
    (panel['ofi_z']     != 0) &
    (np.sign(panel['llm_score']) == np.sign(panel['ofi_z']))
)
confirmed  = panel[same_sign].copy()
conflicted = panel[~same_sign & (panel['llm_score'] != 0) & (panel['ofi_z'] != 0)].copy()

# Compute interaction signal
for sub in [confirmed, conflicted, panel]:
    sub['ofi_x_llm'] = sub['ofi_z'] * sub['llm_score']

print(f'Confirmed events: {len(confirmed):,}')
print(f'Conflicted events: {len(conflicted):,}')
print(f'Confirmed fraction: {len(confirmed)/(len(confirmed)+len(conflicted)):.2%}')

Confirmed events: 5,818
Conflicted events: 5,692
Confirmed fraction: 50.55%


## 3. PRIMARY HYPOTHESIS: confirmed 1-min IC > 0

Reported uncorrected per preregistration (see PREREGISTRATION.md).

In [4]:
primary = bootstrap_ic_ci(confirmed['ofi_x_llm'], confirmed['ret_1m'],
                          n_boot=N_BOOT, seed=42)

print('='*55)
print('PRIMARY HYPOTHESIS (preregistered, uncorrected)')
print(f"  Signal: ofi_x_llm on confirmed events, 1-min horizon")
print(f"  n       = {primary['n']:,}")
print(f"  IC      = {primary['ic']:.4f}")
print(f"  95% CI  = [{primary['ci_lo']:.4f}, {primary['ci_hi']:.4f}]")
print(f"  p_boot  = {primary['p_boot']:.4f}")
if primary['ci_lo'] > 0:
    print('  RESULT: CI entirely above zero — PRIMARY HYPOTHESIS SUPPORTED')
elif primary['ic'] > 0:
    print('  RESULT: IC > 0 but CI crosses zero — marginal / not supported')
else:
    print('  RESULT: IC ≤ 0 — PRIMARY HYPOTHESIS NOT SUPPORTED')
print('='*55)

PRIMARY HYPOTHESIS (preregistered, uncorrected)
  Signal: ofi_x_llm on confirmed events, 1-min horizon
  n       = 5,812
  IC      = -0.0080
  95% CI  = [-0.0333, 0.0176]
  p_boot  = 0.7550
  RESULT: IC ≤ 0 — PRIMARY HYPOTHESIS NOT SUPPORTED


## 4. Confirmed vs conflicted IC table (secondary)

In [5]:
rows = []
for h in HORIZONS:
    ret_col = f'ret_{h}m'
    for label, sub in [('confirmed', confirmed), ('conflicted', conflicted), ('all', panel)]:
        if ret_col not in sub.columns:
            continue
        ci = bootstrap_ic_ci(sub['ofi_x_llm'], sub[ret_col], n_boot=N_BOOT, seed=42)
        rows.append({'split': label, 'horizon': h, **ci})

split_tbl = pd.DataFrame(rows)
display_cols = ['split','horizon','ic','ci_lo','ci_hi','p_boot','n']
print(split_tbl[display_cols].to_string(index=False, float_format='{:.4f}'.format))
save_table(
    split_tbl[display_cols],
    '02_confirmed_conflicted_ic',
    caption='IC for confirmed vs conflicted events (ofi\_x\_llm). CIs: 95\\% bootstrap.',
    label='tab:confirmed_ic',
)

     split  horizon      ic   ci_lo  ci_hi  p_boot     n
 confirmed        1 -0.0080 -0.0333 0.0176  0.7550  5812
conflicted        1  0.0034 -0.0213 0.0302  0.3650  5676
       all        1  0.0054 -0.0120 0.0229  0.2750 12430
 confirmed        5 -0.0020 -0.0264 0.0250  0.5620  5773
conflicted        5 -0.0141 -0.0382 0.0129  0.8470  5632
       all        5 -0.0050 -0.0218 0.0123  0.7040 12342
 confirmed       15 -0.0156 -0.0413 0.0103  0.8900  5675
conflicted       15  0.0158 -0.0124 0.0402  0.1290  5519
       all       15  0.0071 -0.0131 0.0242  0.2260 12115
  Saved table  → results/tables/02_confirmed_conflicted_ic.csv + results/tables/02_confirmed_conflicted_ic.tex


PosixPath('results/tables/02_confirmed_conflicted_ic.csv')

## 5. Conditional OFI regression

In [6]:
reg_rows = []
for scorer in ['llm_score', 'lm_score']:
    try:
        res = fit_all_horizons(panel, delta=0, scorer=scorer, horizons=HORIZONS)
        res['scorer'] = scorer
        reg_rows.append(res)
    except Exception as e:
        print(f'  Regression failed for {scorer}: {e}')

if reg_rows:
    reg_df = pd.concat(reg_rows, ignore_index=True)
    show_cols = ['scorer','horizon','n','b_pos','se_b_pos','b_neg','se_b_neg','wald_p','r2']
    # keep only cols that exist
    show_cols = [c for c in show_cols if c in reg_df.columns]
    print(reg_df[show_cols].to_string(index=False, float_format='{:.4f}'.format))
    save_table(
        reg_df[show_cols],
        '02_conditional_regression',
        caption='Conditional OFI regression: $r_{t+h}=\\alpha+\\gamma\\cdot\\text{OFI}+\\delta_1 s^++\\delta_2 s^-+\\beta_+\\text{OFI}\\cdot s^++\\beta_-\\text{OFI}\\cdot s^-$. HAC SEs. Wald p: $H_0:\\beta_+=\\beta_-$.',
        label='tab:conditional_ofi',
    )

   scorer  horizon     n   b_pos  se_b_pos   b_neg  se_b_neg  wald_p     r2
llm_score        1 12430 -0.0000    0.0000 -0.0000    0.0000  0.7711 0.0002
llm_score        5 12342 -0.0000    0.0001 -0.0001    0.0001  0.4314 0.0004
llm_score       15 12115 -0.0001    0.0001  0.0002    0.0002  0.1696 0.0003
 lm_score        1 12430 -0.0000    0.0000 -0.0000    0.0000  0.7675 0.0007
 lm_score        5 12342 -0.0000    0.0001 -0.0000    0.0001  0.6834 0.0002
 lm_score       15 12115 -0.0000    0.0001 -0.0001    0.0001  0.7741 0.0001
  Saved table  → results/tables/02_conditional_regression.csv + results/tables/02_conditional_regression.tex


## 6. IC decay figure

In [7]:
from src.stats_rigor import ic_decay

ret_cols = {h: f'ret_{h}m' for h in HORIZONS if f'ret_{h}m' in confirmed.columns}

fig, ax = plt.subplots(figsize=(7, 4))

for label, sub, ls in [('confirmed', confirmed, '-'), ('conflicted', conflicted, '--')]:
    avail = {h: sub[c] for h, c in ret_cols.items()}
    if not avail:
        continue
    ics = [spearman_ic(sub['ofi_x_llm'], avail[h]) for h in sorted(avail)]
    ax.plot(sorted(avail.keys()), ics, ls=ls, marker='o', label=label)

ax.axhline(0, color='black', lw=0.7, ls=':', alpha=0.5)
ax.set_xlabel('Horizon (min)')
ax.set_ylabel('Spearman IC')
ax.set_title('IC decay: confirmed vs conflicted events')
ax.legend(frameon=False)
ax.grid(alpha=0.2)
save_fig(fig, '02_ic_decay')
plt.close()

  Saved figure → results/figures/02_ic_decay.pdf


## 7. Append to BH grid

In [8]:
grid_rows = []
for _, row in split_tbl.iterrows():
    if row['split'] == 'confirmed' and row['horizon'] == 1:
        continue  # primary — reported uncorrected
    grid_rows.append({
        'notebook': '02',
        'cell_id':  f"split_{row['split']}_{row['horizon']}m",
        'stock':    'pooled',
        'scorer':   'ofi_x_llm',
        'horizon':  row['horizon'],
        'n':        row['n'],
        'ic':       row['ic'],
        'ci_lo':    row['ci_lo'],
        'ci_hi':    row['ci_hi'],
        'p':        row['p_boot'],
    })

if reg_rows:
    for _, row in reg_df.iterrows():
        if 'wald_p' in row and not pd.isna(row.get('wald_p')):
            grid_rows.append({
                'notebook': '02',
                'cell_id':  f"wald_{row['scorer']}_{row['horizon']}m",
                'stock':    'pooled',
                'scorer':   row['scorer'],
                'horizon':  row['horizon'],
                'n':        row.get('n', np.nan),
                'ic':       np.nan,
                'ci_lo':    np.nan,
                'ci_hi':    np.nan,
                'p':        row['wald_p'],
            })

append_cells(grid_rows)
print(f'Appended {len(grid_rows)} cells to secondary grid.')

Appended 14 cells to secondary grid.
